# 05 — Problem-to-Operator Translation

**Trilha:** Tradução de problemas  
**Milestone:** 3 — Tradução de problemas clássicos para operadores

---

## Pergunta

Como um problema clássico se torna operador + estado inicial + medição? Quais propriedades sobrevivem à tradução? Onde o custo de preparação do estado mata a vantagem?

Este é o notebook diretamente sobre a tese do projeto. O trabalho do engenheiro quântico não é rodar circuitos — é formular o problema na linguagem correta.

## Modelo: os três componentes de uma computação quântica

Todo algoritmo quântico pode ser analisado em três partes:

| Componente | O que é | Custo típico |
|---|---|---|
| **Operador** $U$ ou $H$ | Onde está a estrutura do problema | Depende da estrutura (esparsidade, simetria) |
| **Preparação** $|\psi_0\rangle$ | Estado inicial que codifica a entrada | Muitas vezes exponencialmente caro |
| **Medição** $O$ | O que queremos saber | Depende do observável e da precisão |

**A armadilha comum:** otimizar o operador e ignorar o custo de preparação.

### Padrão de tradução

Para traduzir um problema clássico:
1. Identificar o **espaço de soluções** → espaço de Hilbert
2. Identificar a **função objetivo** → observável ou operador cujos autovalores são relevantes
3. Construir o **estado inicial** → superposição sobre entradas válidas
4. Analisar o **custo de preparação** — aqui frequentemente a vantagem desaparece

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

# --- Caso 1: Estimativa de valor médio ---
# Problema clássico: calcular Tr(ρA) para uma distribuição ρ e observável A
# Tradução quântica: preparar |ψ> que encoda ρ, medir expectation de A

# Exemplo: A é uma matriz diagonal (função em {0,1}^n)
n = 3  # 3 bits -> 8 estados
N = 2**n

# "Função" que queremos estimar a média
f_values = np.array([1, 3, 2, 5, 4, 1, 2, 6], dtype=float)  # f(x) para x=0..7
A = np.diag(f_values)  # observável diagonal: A|x> = f(x)|x>

# Distribuição clássica uniforme -> superposição uniforme
psi_uniform = np.ones(N, dtype=complex) / np.sqrt(N)  # |+>^n

# Média quântica = Média clássica neste caso
quantum_mean = np.real(psi_uniform.conj() @ A @ psi_uniform)
classical_mean = np.mean(f_values)

print('Caso 1: Estimativa de valor médio')
print(f'  f(x) = {f_values}')
print(f'  Média clássica: {classical_mean:.4f}')
print(f'  Média quântica <ψ|A|ψ>: {quantum_mean:.4f}')
print()
print('Tradução:')
print('  Operador: A = diag(f(x)) — O PROBLEMA ESTÁ AQUI')
print('  Preparação: |+>^n = superposição uniforme')
print('  Medição: expectation value de A')
print()
print('Custo de preparação: H^n é O(n) portas. BARATO.')
print('Custo do operador: implementar A eficientemente pode ser CARO se f for complexa.')

In [ ]:
# --- Caso 2: Busca (onde a preparação é trivial, mas a medição exige amplificação) ---
# Problema: encontrar x* tal que f(x*) = 1 em {0,1}^n
# Tradução: oráculo como operador, superposição uniforme, amplificação de amplitude

# Oráculo: inverte fase de |x*>
x_star = 5  # solução
oracle = np.eye(N, dtype=complex)
oracle[x_star, x_star] = -1  # fase kickback na solução

# Verificar: oráculo é unitário
print(f'Oracle unitário: {np.allclose(oracle @ oracle.conj().T, np.eye(N))}')
print(f'Solução codificada: x* = {x_star} (binário: {bin(x_star)})')
print()

# Após 1 aplicação do oráculo: |ψ> -> (-1/N)|x*> + todos os outros
# Sem amplificação: P(x*) = 1/N = 1/8
psi_after_oracle = oracle @ psi_uniform
probs_before_amp = np.abs(psi_after_oracle)**2

print('Probabilidades após 1 oráculo (sem amplificação):')
print(f'  P(x*={x_star}) = {probs_before_amp[x_star]:.4f} = 1/N = {1/N:.4f}')
print(f'  P(outros) = {probs_before_amp[0]:.4f} cada')
print()
print('O oráculo não ajudou diretamente: ainda P(x*) = 1/N.')
print('Grover usa o oráculo + reflexão para AMPLIFICAR P(x*) via interferência.')

In [ ]:
# --- Análise do custo de preparação: quando ele mata a vantagem ---

# Cenário: preparar estado que encoda dados clássicos |D>
# Para encodar N valores em superposição arbitrária: custo O(N) em geral
# (QRAM hipotético: O(log N), mas com constantes enormes)

print('Análise: quando preparação cancela a vantagem')
print('='*55)
print()

# Suponha que o problema tem N=2^n entradas
n_values = [4, 8, 12, 16, 20]

for n_bits in n_values:
    N = 2**n_bits
    
    # Custo clássico de resolução: O(sqrt(N)) — busca clássica
    classical_ops = np.sqrt(N)
    
    # Custo quântico (Grover): O(sqrt(N)) consultas ao oráculo
    quantum_query_complexity = np.sqrt(N)
    
    # Custo de PREPARAR o estado com os dados (sem QRAM): O(N)
    state_prep_cost = N
    
    print(f'n={n_bits:2d} bits (N={N:>7}): '
          f'clássico={classical_ops:>8.0f} | '
          f'Grover (queries)={quantum_query_complexity:>8.0f} | '
          f'prep={state_prep_cost:>7}')

print()
print('Observação: se a preparação do estado custa O(N),')
print('o algoritmo inteiro custa O(N) — sem vantagem sobre busca clássica O(N).')
print()
print('A vantagem de Grover só existe se:')
print('  1. A preparação é eficiente (e.g., superposição uniforme: O(n))')
print('  2. O oráculo é consultado sem precisar preparar o dado novamente')

In [ ]:
# --- Caso 3: Estimativa de fase (VQE-like) ---
# Problema: estimar ground state energy de H
# Tradução: preparar aproximação do ground state, medir <H>

I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)

# Hamiltoniano simples: H = Z + 0.3·X
J = 0.3
H = Z + J * X

# Ground state exato
eigvals, eigvecs = np.linalg.eigh(H)
gs_energy = eigvals[0]
gs_state = eigvecs[:, 0]

print('Caso 3: Estimativa do ground state')
print(f'H = Z + {J}·X')
print(f'Ground state energy (exato): {gs_energy:.6f}')
print()

# Variacional: varrer ângulo θ e medir <H(θ)>
angles = np.linspace(0, np.pi, 100)
expectations = []

for theta in angles:
    psi = np.array([np.cos(theta/2), np.sin(theta/2)], dtype=complex)
    expectations.append(np.real(psi.conj() @ H @ psi))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(angles / np.pi, expectations, color='steelblue')
ax.axhline(gs_energy, color='red', linestyle='--', label=f'Ground state = {gs_energy:.3f}')
ax.set_xlabel('θ/π')
ax.set_ylabel('⟨H⟩')
ax.set_title('Variational energy landscape: mínimo = ground state')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mínimo encontrado: {min(expectations):.6f}')
print()
print('Tradução:')
print('  Operador: H (o Hamiltoniano — é o problema)')
print('  Preparação: |ψ(θ)> parametrizado — CARO se H é grande')
print('  Medição: <H> = soma de expectation values de termos de Pauli')
print()
print('O que se preserva: o mínimo global de <ψ|H|ψ> = ground state energy')
print('O que se perde: convergência garantida ao ground state (VQE tem barrieras clássicas)')

## Conclusão

1. **A tradução sempre tem três partes.** Operador, preparação, medição. Analisar só o operador é analisar 1/3 do problema.

2. **O custo de preparação é frequentemente o gargalo.** Para encodar $N$ dados em superposição genérica, o custo é $O(N)$ — o que elimina qualquer ganho algorítmico.

3. **Preparação barata requer estrutura.** Superposição uniforme: $O(n)$ portas Hadamard. Preparação de estado arbitrário: $O(2^n)$ em geral. A vantagem quântica só sobrevive se a preparação respeita essa restrição.

4. **O que sobrevive à tradução:** a estrutura espectral do operador (autovalores, autovetores). O que se perde: a exatidão da preparação, a eficiência para problemas sem estrutura esparsa, e a garantia de atingir o ótimo em abordagens variacionais.

5. **Conclusão central:** formular o problema para o QPU requer que a estrutura relevante do problema esteja no operador, não na preparação do estado.

---

**Próximo:** a distinção que separa QPE de QSVT — autovalores vs. valores singulares.